## EDA

### 환경 설정

In [2]:
# 데이터 처리 및 분석
import pandas as pd
import ast
import numpy as np
from datetime import datetime, timedelta
import warnings
import re

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
import scikit_posthocs as sp
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import platform

# ── 한글 폰트 설정 ──
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

# ── 출력 설정 ──
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
np.random.seed(42)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("=" * 60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [3]:
# 데이터 로드
df = pd.read_csv('./final_data/master_table.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1181655 entries, 0 to 1181654
Data columns (total 31 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   collect_date           1181655 non-null  str    
 1   platform               1181655 non-null  str    
 2   review_id              1181655 non-null  float64
 3   product_id             1181655 non-null  object 
 4   product_name           1180129 non-null  str    
 5   review_date            1181655 non-null  str    
 6   year                   1181655 non-null  int64  
 7   month                  1181655 non-null  int64  
 8   content                1181655 non-null  str    
 9   rating                 1181655 non-null  int64  
 10  helpful_count          1181655 non-null  int64  
 11  has_image              1181655 non-null  int64  
 12  purchase_option        1044916 non-null  str    
 13  purchase_option_color  928581 non-null   str    
 14  purchase_option_size   961340

### 결측치 처리

In [4]:
print('[브랜드별 리뷰 수]')
print(df['brand'].value_counts(dropna=False).to_string())

[브랜드별 리뷰 수]
brand
안다르     637529
젝시믹스    501893
FILA     20063
룰루레몬     11460
NaN      10710


In [5]:
# 브랜드 NaN 값 확인
df[df['brand'].isnull()]

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
98570,2026-04-21,자사몰,48317460.0,9012,[아울렛] 주니어 에어리핏 스탠다드핏 롱슬리브 (여),2026-03-28,2026,3,세일해서 구매했어요 아직 많이 크지만 내후년에도 입을 수 있을 것 같아요 왜냐하면 ...,5,0,1,"130, [JMPLS-05BLK] 블랙",블랙,130,0,NaN,NaN,139cm 이하,44kg 이하,https://andar.co.kr/product/detail.html?produc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98571,2026-04-21,자사몰,48296778.0,9012,[아울렛] 주니어 에어리핏 스탠다드핏 롱슬리브 (여),2026-03-26,2026,3,안다르 상의 하의 다 입고 등교했어요 오늘 줄넘기 학원 가는지라 가볍고 편하게 입혔...,4,0,1,"140, [JMPLS-05BLK] 블랙",블랙,140,0,NaN,NaN,139cm 이하,44kg 이하,https://andar.co.kr/product/detail.html?produc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98572,2026-04-21,자사몰,48081589.0,9012,[아울렛] 주니어 에어리핏 스탠다드핏 롱슬리브 (여),2026-03-19,2026,3,재질이 고급스럽게 시원하면서도 신축성이 있어서 좋아요~ 외국에 있는 손녀가 좋아하...,5,0,1,"140, [JMPLS-05LPK] 코랄클라우드",코랄클라우드,140,0,NaN,NaN,139cm 이하,44kg 이하,https://andar.co.kr/product/detail.html?produc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98573,2026-04-21,자사몰,47747998.0,9012,[아울렛] 주니어 에어리핏 스탠다드핏 롱슬리브 (여),2026-03-16,2026,3,소재가 들러붙지 않고 쾌적하니 좋네요\n탄탄해서 세탁후 변형 걱정도 없어서 자주 입...,5,0,1,"140, [JMPLS-05LPK] 코랄클라우드",코랄클라우드,140,0,NaN,NaN,155~159cm,55~59kg,https://andar.co.kr/product/detail.html?produc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98574,2026-04-21,자사몰,46709066.0,9012,[아울렛] 주니어 에어리핏 스탠다드핏 롱슬리브 (여),2026-01-27,2026,1,"가볍고 쭉쭉 늘어나니까,\n“체육 시간 있는 날도\n아이가 먼저 챙겨 입어요” [아...",5,0,1,"130, [JMPLS-05BLK] 블랙",블랙,130,0,NaN,NaN,139cm 이하,44kg 이하,https://andar.co.kr/product/detail.html?produc...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1181545,2026-04-21 23:30:06,무신사,83070488.0,5886275,리브드 드레이피 Softstreme 롱슬리브 튜닉 BLK,2026-03-19,2026,3,좀 무겁지만 색감이랑 디자인이 맘에 들어요 세일가격도 맘에 들고요 데일리룩으로도 수...,5,0,1,black · xxs,블랙,090,0,158.0,45.0,155~159cm,45~49kg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1181547,2026-04-21 23:30:15,무신사,82239935.0,5879373,LuluLinen 릴랙스 핏 버튼업 셔츠 WHT,2026-02-13,2026,2,세일해서 구매했어요! s사이즈 샀다가 애매하게 큰 느낌이라 m으로 교환하니까 딱 좋아요,5,0,1,white · m,화이트,100,0,160.0,53.0,160~164cm,50~54kg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1181548,2026-04-21 23:30:20,무신사,83768410.0,5879371,LuluLinen 릴랙스 핏 버튼업 셔츠 DVGY,2026-04-18,2026,4,소재 시원하고 차분한 그레이네요 툭 걸치기 좋습니다,5,0,0,dove grey · s,도브_그레이,095,0,167.0,58.0,165~169cm,55~59kg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1181551,2026-04-21 23:30:35,무신사,83190340.0,5879301,LuluLinen 타이 프런트 재킷 DVGY,2026-03-24,2026,3,기대 이상으로 만족스러운 제품이에요 정말로 너무 좋아요,5,0,0,dove grey · s,도브_그레이,095,0,177.0,65.0,175~179cm,65~69kg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# 브랜드 NaN 값 제거
before = len(df)
df = df.dropna(subset=['brand'])
after = len(df)
print(f'제거 전 리뷰 수: {before}, 제거 후 리뷰 수: {after}')

제거 전 리뷰 수: 1181655, 제거 후 리뷰 수: 1170945


##### 리뷰는 있지만 상품정보는 없는 데이터여서 삭제 진행

In [7]:
df.isnull().sum()

collect_date                  0
platform                      0
review_id                     0
product_id                    0
product_name                  0
review_date                   0
year                          0
month                         0
content                       0
rating                        0
helpful_count                 0
has_image                     0
purchase_option          134565
purchase_option_color    250285
purchase_option_size     218135
women_size_yn                 0
user_height              724072
user_weight              731984
user_height_group        253321
user_weight_group        263137
product_url               11460
brand                         0
cat1                         39
cat2                         39
cat3                          0
gender                        0
original_price                0
discount_price                0
color_count                   0
is_new                        0
is_best                       0
dtype: i

In [8]:
df[df['cat1'].isnull()]

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,helpful_count,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url,brand,cat1,cat2,cat3,gender,original_price,discount_price,color_count,is_new,is_best
1180126,2026-04-21 22:57:48,무신사,83547450.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-09,2026,4,룰루레몬 헤어밴드 좋습니다 러닝할때 아주 잘 사용해요,4,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180127,2026-04-21 22:57:48,무신사,83449647.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,땀이 많은 편이라 러닝 한번 하고나면 상의가 반쯤 젖을만큼 땀을 흘리는데 확실히 밴...,5,0,0,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180128,2026-04-21 22:57:48,무신사,83439220.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,안벗겨지고 깔끔하고 무난합니다 땀도 잘 흡수하고 가볍고 빨리 마르고 착용감도 너어무...,5,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180129,2026-04-21 22:57:48,무신사,83189716.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-04,2026,4,벌써 여러 가지 색상으로 룰루레몬에 헤어밴드를 많이 가지고 있습니다 매우 만족하며 ...,5,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180130,2026-04-21 22:57:48,무신사,83375899.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-04-01,2026,4,운동할 때 쓰려고 샀는데 진짜 너무 좋아요 재질도 너무 만족스럽네요,5,0,0,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180131,2026-04-21 22:57:48,무신사,83271423.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-03-28,2026,3,좋음 가성비 좋음 룰루레몬 룰루랄라랄 탄력 굳 땀흡수 굳,5,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180132,2026-04-21 22:57:48,무신사,83211277.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-03-25,2026,3,땀흡수 잘되고 빨리 마릅니다 손빨래후에도 금방 마르고 넓이감이 있어서 넓게쓰거나 접...,5,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180133,2026-04-21 22:57:48,무신사,83112865.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-03-21,2026,3,소재도 좋고 짱짱하네요! 러닝할때 사용하려고 삿는데 세일할때 잘 삿습니다,5,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180134,2026-04-21 22:57:48,무신사,83101955.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-03-20,2026,3,배송도 빠르고 재질도 좋아요 사이즈도 프리사이즈라 누구나 잘맞겠네요,4,0,1,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False
1180135,2026-04-21 22:57:48,무신사,83081690.0,5733085,심리스 와이드 헤드밴드 SOGY VPOR,2026-03-19,2026,3,소재가 좋음 가볍고 땀흡수 잘됨 하지만 나한텐 잘 안어올림,4,0,0,solar grey / vapor · one size,"솔라_그레이,베이퍼",000,0,NaN,NaN,NaN,NaN,NaN,룰루레몬,NaN,NaN,트레이닝,Unisex,29000.0,29000.0,8.0,False,False


In [9]:
# cat1의 NaN 값을 '용품'으로 대체
df['cat1'] = df['cat1'].fillna('용품')
# cat2의 NaN 값을 '기타'로 대체
df['cat2'] = df['cat2'].fillna('기타')

df['cat1'].isnull().sum(), df['cat2'].isnull().sum()

(np.int64(0), np.int64(0))

In [10]:
df.isnull().sum()

collect_date                  0
platform                      0
review_id                     0
product_id                    0
product_name                  0
review_date                   0
year                          0
month                         0
content                       0
rating                        0
helpful_count                 0
has_image                     0
purchase_option          134565
purchase_option_color    250285
purchase_option_size     218135
women_size_yn                 0
user_height              724072
user_weight              731984
user_height_group        253321
user_weight_group        263137
product_url               11460
brand                         0
cat1                          0
cat2                          0
cat3                          0
gender                        0
original_price                0
discount_price                0
color_count                   0
is_new                        0
is_best                       0
dtype: i

In [11]:
# 범주형 - unknown으로 대체
fill_cols = [
    'purchase_option', 'purchase_option_color', 'purchase_option_size',
    'user_height_group', 'user_weight_group'
]
df[fill_cols] = df[fill_cols].fillna('unknown')

# 수치형 - NaN 유지 (dtype 보존)
# user_height, user_weight는 분석 시 아래처럼 사용
# df.dropna(subset=['user_height', 'user_weight'])
df.info()

<class 'pandas.DataFrame'>
Index: 1170945 entries, 0 to 1181654
Data columns (total 31 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   collect_date           1170945 non-null  str    
 1   platform               1170945 non-null  str    
 2   review_id              1170945 non-null  float64
 3   product_id             1170945 non-null  object 
 4   product_name           1170945 non-null  str    
 5   review_date            1170945 non-null  str    
 6   year                   1170945 non-null  int64  
 7   month                  1170945 non-null  int64  
 8   content                1170945 non-null  str    
 9   rating                 1170945 non-null  int64  
 10  helpful_count          1170945 non-null  int64  
 11  has_image              1170945 non-null  int64  
 12  purchase_option        1170945 non-null  str    
 13  purchase_option_color  1170945 non-null  str    
 14  purchase_option_size   1170945 non

In [12]:
# 브랜드별 데이터 추출 및 저장
brands = df['brand'].unique()
for brand in brands:
    brand_df = df[df['brand'] == brand]
    filename = f'./final_data/{brand}_master.csv'
    brand_df.to_csv(filename, index=False)
    print(f'{brand} 마스터 데이터 저장 완료: {filename}')

# 기초 전처리 완료된 마스터 테이블 저장
df.to_csv('./final_data/master_preprocessed_260430.csv', index=False)

안다르 마스터 데이터 저장 완료: ./final_data/안다르_master.csv
FILA 마스터 데이터 저장 완료: ./final_data/FILA_master.csv
젝시믹스 마스터 데이터 저장 완료: ./final_data/젝시믹스_master.csv
룰루레몬 마스터 데이터 저장 완료: ./final_data/룰루레몬_master.csv


In [13]:
print(f'전체 마스터테이블: {len(df):,}행 × {len(df.columns)}컬럼')

전체 마스터테이블: 1,170,945행 × 31컬럼


In [14]:
df.shape

(1170945, 31)